# 行星半径物理合理性校验 (Planet Radius Plausibility)

**目的**：承接奇偶校验的输出，对每个候选换算行星绝对半径，剔除"深度稳定但天体过大"的假信号(恒星伴星/褐矮星)——这类信号奇偶校验抓不到。

**物理原理**：
- 凌日深度只给出半径比：$R_p/R_\star=\sqrt{\delta}$（$\delta$=深度分数）
- 必须乘以宿主恒星真实半径 $R_\star$ 才得到行星绝对半径：$R_p=\sqrt{\delta}\times R_\star$
- 天体半径有物理天花板(简并压)：木星以上半径基本不再增大，行星半径上限约 2~2.5 Rjup；超过即为褐矮星/恒星
- **M 型矮星**盘质量小，几乎不可能有远超木星的巨行星，上限更严

**流程**：
```
① astroquery 查 TIC/Gaia  取宿主恒星半径 R* 与有效温度 Teff
② 换算半径               Rp = sqrt(深度) × R*   (转成木星/地球半径单位)
③ 定光谱型               由 Teff 分档 O/B/A/F/G/K/M
④ 判定                   Rp > 2.5 Rjup(M矮星 2.0) → 不合理；查不到 R* → 缺失
```

**输入**：优先用奇偶校验输出 `odd_even_result.csv`，回退 Person1 的 `master_candidates.csv`。
**输出**：新增 `planet_radius_plausible` 布尔列(及诊断列)，保存为 `radius_result.csv`，不改动 Person1 原始字段。

## 0. 安装依赖

In [1]:
%pip install -q numpy pandas astropy astroquery

Note: you may need to restart the kernel to use updated packages.


## 1. 配置与加载数据

优先读取上一步(奇偶校验)的输出，以便三项校验的结果逐步累积在同一张表里。

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

# ---------------- 物理常数与阈值 ----------------
R_SUN_TO_RJUP   = 9.7311      # 1 太阳半径 = 9.73 木星半径
R_JUP_TO_REARTH = 11.2089     # 1 木星半径 = 11.21 地球半径
MAX_RP_RJUP_GENERAL = 2.5     # 通用上限：>2.5 Rjup 进入褐矮星/恒星范畴
MAX_RP_RJUP_MDWARF  = 2.0     # M 矮星：巨行星极罕见，上限更严

# ---------------- 输入/输出 ----------------
# 优先承接奇偶校验输出；逐级回退到 Person1 原始表
_PATHS = ['odd_even_result.csv',
          'outputs/candidates_annotated.csv',
          'master_candidates.csv',
          'attachments/master_candidates.csv']
INPUT_CSV = next((p for p in _PATHS if os.path.exists(p)), _PATHS[-1])
OUTPUT_CSV = 'radius_result.csv'

# Person1 原始 8 列(用于完整性校验，确保永不改动)
PERSON1_COLS = ['Target', 'Sector', 'Period (d)', 'Period Err (d)',
                'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']

df = pd.read_csv(INPUT_CSV)
df_in = df.copy()                 # 输入快照，用于末尾一致性校验
print(f'已加载 {INPUT_CSV} -> {len(df)} 个候选')
print('现有列:', list(df.columns))
df.head()

已加载 odd_even_result.csv -> 50 个候选
现有列: ['Target', 'Sector', 'Period (d)', 'Period Err (d)', 'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive', 'depth_odd_ppm', 'depth_even_ppm', 'n_transits', 'odd_even_sigma', 'odd_even_reason', 'odd_even_pass']


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive,depth_odd_ppm,depth_even_ppm,n_transits,odd_even_sigma,odd_even_reason,odd_even_pass
0,L 98-59,2,3.68989,0.00055,685.0,1.3,13.43,False,1227.087681,1367.776127,6,1.307202,ok,True
1,TOI-700,1,11.34365,0.28837,1283.0,0.5,5.10,True,3275.255154,1012.759695,3,4.330612,ok,False
2,TOI-270,3,11.32061,0.12267,1531.5,1.3,10.14,False,2976.994045,2304.645538,2,2.336046,ok,True
3,GJ 357,8,9.27961,0.18019,820.3,0.5,8.37,False,1444.105532,1026.678471,2,2.424015,ok,True
4,HD 21749,1,22.13134,0.69646,4033.4,0.5,17.64,False,0.000000,1393.418770,1,NaN,oddeven_err_undefined(n_transits=1),NaN


## 2. 核心函数

- `get_stellar_params`：从 TIC 查恒星半径/温度/质量与 TIC ID，返回 `note` 标注查询结果(ok / no_stellar_radius / no_tic_match / exception)。
- `classify_spectral`：由有效温度粗分光谱型。
- `assess_radius`：换算 $R_p$ 并判定合理性，同时给出可读原因。

In [3]:
from astroquery.mast import Catalogs

_TIC_CACHE = {}

def _f(x):
    '''安全地把可能被 mask 的表元素转为 float。'''
    try:
        if x is None or np.ma.is_masked(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def get_stellar_params(target):
    '''从 TIC(整合了 Gaia 参数)查询宿主恒星：半径(R_sun)、Teff(K)、质量、TIC ID。'''
    if target in _TIC_CACHE:
        return _TIC_CACHE[target]
    out = {'tic_id': None, 'rad': np.nan, 'teff': np.nan, 'mass': np.nan, 'note': 'query_failed'}
    try:
        cat = Catalogs.query_object(str(target), catalog='TIC', radius=0.02)
        if cat is not None and len(cat) > 0:
            try:                                    # 取最亮者作为宿主星
                order = np.argsort([_f(v) for v in cat['Tmag']])
                cat = cat[order]
            except Exception:
                pass
            r = cat[0]
            out['tic_id'] = int(r['ID'])
            out['rad']  = _f(r['rad'])
            out['teff'] = _f(r['Teff'])
            out['mass'] = _f(r['mass'])
            out['note'] = 'ok' if np.isfinite(out['rad']) else 'no_stellar_radius'
        else:
            out['note'] = 'no_tic_match'
    except Exception as e:
        out['note'] = f'exception:{type(e).__name__}'
    _TIC_CACHE[target] = out
    return out

def classify_spectral(teff):
    '''按有效温度粗分光谱型。'''
    if not np.isfinite(teff):
        return 'Unknown'
    if teff >= 30000: return 'O'
    if teff >= 10000: return 'B'
    if teff >= 7500:  return 'A'
    if teff >= 6000:  return 'F'
    if teff >= 5200:  return 'G'
    if teff >= 3700:  return 'K'
    if teff >= 2400:  return 'M'
    return 'L/T(次恒星)'

def assess_radius(depth_ppm, rstar_rsun, teff):
    '''返回 (合理 or pd.NA, Rp_Rjup, Rp_Rearth, 光谱型, 原因)。'''
    spectral = classify_spectral(teff)
    d = float(depth_ppm) * 1e-6
    if not np.isfinite(d) or d <= 0:
        return pd.NA, np.nan, np.nan, spectral, 'bad_depth'
    if not np.isfinite(rstar_rsun):
        return pd.NA, np.nan, np.nan, spectral, 'no_stellar_radius'
    rp_rjup = np.sqrt(d) * rstar_rsun * R_SUN_TO_RJUP
    rp_rearth = rp_rjup * R_JUP_TO_REARTH
    max_rp = MAX_RP_RJUP_MDWARF if spectral == 'M' else MAX_RP_RJUP_GENERAL
    if rp_rjup <= max_rp:
        return True, rp_rjup, rp_rearth, spectral, 'plausible'
    reason = 'too_large_mdwarf' if spectral == 'M' else 'too_large'
    return False, rp_rjup, rp_rearth, spectral, reason

## 2.1 单目标诊断（先跑这一格）

用第一个候选验证 astroquery 能否连通 TIC 并返回恒星半径。若返回 `no_tic_match`/`exception`，多为网络问题；`no_stellar_radius` 表示查到了星但缺半径，属数据缺失。

In [4]:
_t = str(df['Target'].iloc[0])
_sp = get_stellar_params(_t)
print('宿主恒星查询:', _t, '->', _sp)
_depth = float(df['Depth (ppm)'].iloc[0])
_ok, _rj, _re, _spec, _rsn = assess_radius(_depth, _sp['rad'], _sp['teff'])
print(f'深度={_depth}ppm  光谱型={_spec}  Rp={_rj if np.isfinite(_rj) else "NA"} Rjup '
      f'({_re if np.isfinite(_re) else "NA"} R⊕)  合理={_ok}  原因={_rsn}')
if _sp['note'] in ('no_tic_match',) or _sp['note'].startswith('exception'):
    print('>> 提示：TIC 查询未命中或报错，检查网络/astroquery 访问。')
elif _sp['note'] == 'ok':
    print('>> 链路正常，可继续批量。')

宿主恒星查询: L 98-59 -> {'tic_id': 307210830, 'rad': 0.31416, 'teff': 3429.0, 'mass': 0.292836, 'note': 'ok'}
深度=685.0ppm  光谱型=M  Rp=0.08001254962167073 Rjup (0.896852667454345 R⊕)  合理=True  原因=plausible
>> 链路正常，可继续批量。


## 3. 批量半径校验

In [5]:
tic_list, rstar_list, teff_list = [], [], []
spec_list, rj_list, re_list = [], [], []
plausible_list, reason_list = [], []

for i, row in df.iterrows():
    sp = get_stellar_params(row['Target'])
    ok, rj, re, spec, reason = assess_radius(row['Depth (ppm)'], sp['rad'], sp['teff'])
    tic_list.append(sp['tic_id']); rstar_list.append(sp['rad']); teff_list.append(sp['teff'])
    spec_list.append(spec); rj_list.append(rj); re_list.append(re)
    plausible_list.append(ok); reason_list.append(reason)
    rj_txt = 'NaN' if not np.isfinite(rj) else f'{rj:.2f}'
    rstar_txt = 'NA' if not np.isfinite(sp['rad']) else f"{sp['rad']:.3f}"
    print(f"[{i+1:>2}/{len(df)}] {str(row['Target']):<12} R*={rstar_txt:<6} 型={spec:<8} "
          f"Rp={rj_txt:<6}Rjup  合理={str(ok):<5} 原因={reason}")

df['TIC_ID']              = tic_list
df['Stellar_Radius_Rsun'] = rstar_list
df['Teff_K']              = teff_list
df['Spectral_Type']       = spec_list
df['Rp_Rjup']             = rj_list
df['Rp_Rearth']           = re_list
df['radius_reason']       = reason_list
df['planet_radius_plausible'] = pd.array(plausible_list, dtype='boolean')

print('\n半径校验完成 ->  合理:', int((df['planet_radius_plausible'] == True).sum()),
      ' 不合理:', int((df['planet_radius_plausible'] == False).sum()),
      ' 无法判定:', int(df['planet_radius_plausible'].isna().sum()))
print('原因分布:')
for reason, cnt in Counter(reason_list).most_common():
    print(f'   {cnt:>3}  x  {reason}')

[ 1/50] L 98-59      R*=0.314  型=M        Rp=0.08  Rjup  合理=True  原因=plausible
[ 2/50] TOI-700      R*=0.419  型=M        Rp=0.15  Rjup  合理=True  原因=plausible
[ 3/50] TOI-270      R*=0.374  型=M        Rp=0.14  Rjup  合理=True  原因=plausible
[ 4/50] GJ 357       R*=0.360  型=M        Rp=0.10  Rjup  合理=True  原因=plausible
[ 5/50] HD 21749     R*=0.705  型=K        Rp=0.44  Rjup  合理=True  原因=plausible
[ 6/50] TOI-125      R*=0.852  型=G        Rp=0.23  Rjup  合理=True  原因=plausible
[ 7/50] TOI-396      R*=1.284  型=F        Rp=0.17  Rjup  合理=True  原因=plausible
[ 8/50] TOI-451      R*=0.856  型=G        Rp=0.34  Rjup  合理=True  原因=plausible
[ 9/50] TOI-561      R*=0.841  型=G        Rp=0.20  Rjup  合理=True  原因=plausible
[10/50] TOI-620      R*=8.636  型=K        Rp=4.91  Rjup  合理=False 原因=too_large
[11/50] LTT 1445     R*=0.385  型=Unknown  Rp=0.10  Rjup  合理=True  原因=plausible
[12/50] TOI-776      R*=0.533  型=K        Rp=0.14  Rjup  合理=True  原因=plausible
[13/50] TOI-824      R*=0.730  型=K        Rp=0.18  R

## 4. 保存结果

In [6]:
# 完整性校验：Person1 原始列未被改动
_p1 = [c for c in PERSON1_COLS if c in df.columns and c in df_in.columns]
assert df[_p1].equals(df_in[_p1]), 'Person1 原始数据被意外修改!'
print('原始数据完整性校验通过。\n')

df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print('已保存 ->', OUTPUT_CSV)

# 展示被判为不合理(疑似恒星/褐矮星)的目标
bad = df[df['planet_radius_plausible'] == False]
print(f'\n判定不合理的目标: {len(bad)} 个')
if len(bad) > 0:
    print(bad[['Target', 'Depth (ppm)', 'Stellar_Radius_Rsun', 'Spectral_Type',
               'Rp_Rjup', 'radius_reason']].to_string(index=False))
df

原始数据完整性校验通过。

已保存 -> radius_result.csv

判定不合理的目标: 1 个
 Target  Depth (ppm)  Stellar_Radius_Rsun Spectral_Type  Rp_Rjup radius_reason
TOI-620       3411.5              8.63601             K 4.908488     too_large


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive,depth_odd_ppm,depth_even_ppm,n_transits,odd_even_sigma,odd_even_reason,odd_even_pass,TIC_ID,Stellar_Radius_Rsun,Teff_K,Spectral_Type,Rp_Rjup,Rp_Rearth,radius_reason,planet_radius_plausible
0,L 98-59,2,3.68989,0.00055,685.0,1.30,13.43,False,1227.087681,1367.776127,6,1.307202,ok,True,307210830,0.314160,3429.00,M,0.080013,0.896853,plausible,True
1,TOI-700,1,11.34365,0.28837,1283.0,0.50,5.10,True,3275.255154,1012.759695,3,4.330612,ok,False,150428135,0.419457,3494.00,M,0.146205,1.638799,plausible,True
2,TOI-270,3,11.32061,0.12267,1531.5,1.30,10.14,False,2976.994045,2304.645538,2,2.336046,ok,True,259377017,0.374358,3532.00,M,0.142563,1.597977,plausible,True
3,GJ 357,8,9.27961,0.18019,820.3,0.50,8.37,False,1444.105532,1026.678471,2,2.424015,ok,True,413248763,0.360144,3490.00,M,0.100375,1.125091,plausible,True
4,HD 21749,1,22.13134,0.69646,4033.4,0.50,17.64,False,0.000000,1393.418770,1,NaN,oddeven_err_undefined(n_transits=1),NaN,279741379,0.705464,4628.53,K,0.435986,4.886922,plausible,True
5,TOI-125,1,9.19402,0.16300,737.8,0.50,4.40,True,938.750423,534.932163,3,1.409259,ok,True,52368076,0.852266,5280.00,G,0.225272,2.525047,plausible,True
6,TOI-396,4,19.17518,0.53385,177.2,0.50,7.48,False,341.385111,137.080858,2,2.861460,ok,True,178155732,1.284500,6308.31,F,0.166390,1.865052,plausible,True
7,TOI-451,4,12.13043,0.28565,1659.1,1.30,8.10,False,0.000000,1657.541891,2,NaN,oddeven_err_undefined(n_transits=2),NaN,257605131,0.856226,5481.00,G,0.339380,3.804080,plausible,True
8,TOI-561,8,13.94758,0.40304,589.3,0.50,4.11,True,952.046491,502.225149,2,1.841273,ok,True,377064495,0.840576,5440.00,G,0.198567,2.225718,plausible,True
9,TOI-620,8,12.05471,0.30074,3411.5,1.30,16.32,False,0.000000,6490.738704,1,NaN,oddeven_err_undefined(n_transits=1),NaN,296739875,8.636010,4872.00,K,4.908488,55.018754,too_large,False


## 5. 说明

- `planet_radius_plausible`：`True`=半径在物理上限内，`False`=Rp 超上限(疑似恒星/褐矮星)，`<NA>`=查不到恒星半径无法判定。
- 阈值 2.5 / 2.0 Rjup 为经验上限，可在配置区调整；文献中也常用 ~8 R⊕(褐矮星下限)作为可验证行星的半径上限。
- 保守原则：`no_stellar_radius`/`no_tic_match` 记为缺失而非通过。
- 本步是奇偶校验的互补关卡：奇偶查"深度忽深忽浅"的双星，半径查"深度稳定但天体过大"的恒星伴星。
- 输出 `radius_result.csv` 已累积奇偶+半径两项校验结果，可直接进入第三步 NASA 档案交叉比对。